# Exercises XP – Week 5, Day 2: Heart Disease Prediction
Comparing **Logistic Regression, SVM and XGBoost**, each without and with **Grid Search**.
Dataset: `dataset_heart.csv` (270 patients). Needs `xgboost` (pre-installed in Colab; locally: `pip install xgboost`).

## 🌟 Exercise 1: Exploratory Data Analysis

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

warnings.filterwarnings("ignore", module="sklearn")   # newer scikit-learn versions warn about the penalty= parameter
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

LOCAL_FILE = "dataset_heart.csv"
GITHUB_URL = "https://raw.githubusercontent.com/jenny-palti/DI-Bootcamp2026/main/Week5/Day2/dataset_heart.csv"
df = pd.read_csv(LOCAL_FILE if os.path.exists(LOCAL_FILE) else GITHUB_URL)

df.columns = df.columns.str.strip()      # some column names have extra spaces, e.g. "sex "
print(df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\nTarget values:", df["heart disease"].value_counts().to_dict())

The target `heart disease` is coded **1 = absent, 2 = present**. We convert it to **0 / 1** (XGBoost requires classes starting at 0).

In [ ]:
df["heart disease"] = df["heart disease"].map({1: 0, 2: 1})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x="heart disease", ax=axes[0], hue="heart disease", palette="Set2", legend=False)
axes[0].set_title("Target: 0 = no disease, 1 = disease")
sns.boxplot(data=df, x="heart disease", y="max heart rate", ax=axes[1])
axes[1].set_title("Max heart rate by target")
sns.boxplot(data=df, x="heart disease", y="age", ax=axes[2])
axes[2].set_title("Age by target")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

df.corr()["heart disease"].drop("heart disease").sort_values(key=abs, ascending=False).round(2)

**Understanding the data:** 270 patients, 13 numeric features, no missing values, and fairly **balanced** classes (150 without vs. 120 with heart disease). The features most related to heart disease are **thal**, **number of major vessels**, **chest pain type**, **exercise-induced angina**, **oldpeak** and **max heart rate** (lower max heart rate → more disease).

In [ ]:
# Remove the target column from the features, then split into train / test
X = df.drop(columns="heart disease")
y = df["heart disease"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

def evaluate(name, model):
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    results[name] = acc
    print(f"{name} – test accuracy: {acc:.3f}")
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, digits=3))

## 🌟 Exercise 2: Logistic Regression without Grid Search

In [ ]:
# Scaling is part of the pipeline, so it's fitted on the training data only
log_reg = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
log_reg.fit(X_train, y_train)
evaluate("LogReg (default)", log_reg)

## 🌟 Exercise 3: Logistic Regression with Grid Search

In [ ]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"],          # liblinear supports both l1 and l2
}
grid_lr = GridSearchCV(Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
                       param_grid, cv=cv, scoring="accuracy")
grid_lr.fit(X_train, y_train)
print("Best parameters:", grid_lr.best_params_)
print("Best cross-validated accuracy:", round(grid_lr.best_score_, 3))
evaluate("LogReg (grid search)", grid_lr)

## 🌟 Exercise 4: SVM without Grid Search
I choose an **RBF kernel** (it can capture non-linear relationships) with the standard settings `C=1` and `gamma='scale'`. SVMs depend on distances, so the data must be scaled.

In [ ]:
svm = Pipeline([("scale", StandardScaler()), ("model", SVC(kernel="rbf", C=1, gamma="scale"))])
svm.fit(X_train, y_train)
evaluate("SVM (manual)", svm)

## 🌟 Exercise 5: SVM with Grid Search

In [ ]:
param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf", "poly"],
    "model__gamma": ["scale", 0.001, 0.01, 0.1],
}
grid_svm = GridSearchCV(Pipeline([("scale", StandardScaler()), ("model", SVC())]),
                        param_grid, cv=cv, scoring="accuracy")
grid_svm.fit(X_train, y_train)
print("Best parameters:", grid_svm.best_params_)
print("Best cross-validated accuracy:", round(grid_svm.best_score_, 3))
evaluate("SVM (grid search)", grid_svm)

## 🌟 Exercise 6: XGBoost without Grid Search
Manual choices and why:
- `n_estimators=100`: enough trees for a small dataset without taking long.
- `max_depth=3`: shallow trees, because with only 216 training rows deep trees would overfit.
- `learning_rate=0.1`: the common default, a good balance between speed and stability.
- `subsample=0.8`, `colsample_bytree=0.8`: each tree sees 80% of rows and features, which adds randomness and reduces overfitting.

Tree models don't need feature scaling.

In [ ]:
xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42)
xgb.fit(X_train, y_train)
evaluate("XGBoost (manual)", xgb)

## 🌟 Exercise 7: XGBoost with Grid Search

In [ ]:
param_grid = {
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "n_estimators": [50, 100, 200],
    "max_depth": [2, 3, 4, 5],
    "subsample": [0.8, 1.0],
}
grid_xgb = GridSearchCV(XGBClassifier(eval_metric="logloss", random_state=42),
                        param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
grid_xgb.fit(X_train, y_train)
print("Best parameters:", grid_xgb.best_params_)
print("Best cross-validated accuracy:", round(grid_xgb.best_score_, 3))
evaluate("XGBoost (grid search)", grid_xgb)

## Comparison of Model Performances

In [ ]:
comparison = pd.Series(results).sort_values()
comparison.plot(kind="barh", figsize=(8, 4), color="steelblue")
plt.xlim(0.6, 1.0)
for i, v in enumerate(comparison):
    plt.text(v + 0.005, i, f"{v:.3f}", va="center")
plt.title("Test accuracy by model")
plt.show()
comparison.sort_values(ascending=False).round(3)

## Analysis Report
- **All models perform in a similar range** on this small dataset (270 patients, 54 in the test set). With so few test patients, **one patient changes the accuracy by about 1.9 percentage points**, so small differences between models are not meaningful. The **cross-validated scores** printed by the grid searches are a more reliable basis for comparison than one test split.
- **Logistic regression is a very strong baseline** here: the relationships between the features and heart disease are fairly linear, and the simple model is easy to interpret for doctors.
- **Grid search does not always improve the test score.** It picks the settings that are best on average across the 5 training folds, which protects against an unlucky choice, but on a tiny test set the "tuned" model can score the same as, or even slightly below, the default one. That's not a failure, just noise.
- **XGBoost** is the most flexible model, but with only ~216 training rows it can easily overfit; shallow trees and a moderate learning rate (which the grid search typically selects) work best.
- **Model selection:** I would pick the model with the best **cross-validated** score. If it's close, I'd prefer the **simplest, most explainable** one (logistic regression), because in a medical context understanding *why* a patient is flagged matters. I'd also check **recall** for the disease class, since missing a sick patient is the most costly error.